# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports a checkpoint to `/kaggle/working`.

Current experiment: **E001-pipe-check-gold58** — frozen backbone + linear head on the
58 gold-labeled studies (issue #6). Requires the `WANDB_API_KEY` Kaggle secret.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
COMMIT = "main"  # TODO: pin to a SHA per run
%pip install -q "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee[train]"

from knee.train_gold import train_gold

In [ ]:
# Gold-58 prototype trains straight off the mounted competition data; the private
# mined-labels dataset joins here later (issue #2).
from pathlib import Path

COMP_ROOT = Path("/kaggle/input/rsna-knee-abnormality-detection")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
run = wandb.init(project="rsna-knee", config={"commit": COMMIT})

In [ ]:
# E001-pipe-check-gold58: this checkpoint trains on the gold studies and must never
# be evaluated against them.
CHECKPOINT = Path("/kaggle/working/pipe_check_gold58.pt")
INPUT_SIZE = 224

In [ ]:
# Competition DICOMs are pre-mounted read-only; ~58 series to decode, minutes of GPU.
result = train_gold(COMP_ROOT, CHECKPOINT, input_size=INPUT_SIZE)
print(f"trained on {result.n_studies} studies, skipped {len(result.skipped)}")
# In-sample only (trains on all 58): proves the features carry signal, nothing more.
print(result.in_sample_auc)

In [ ]:
# Checkpoint is already in /kaggle/working, which persists as notebook output;
# publish it as the knee-weights dataset so the inference notebook can attach it.
import math

wandb.log({f"in_sample_auc/{k}": v for k, v in result.in_sample_auc.items() if not math.isnan(v)})
run.finish()